# DEEZER API PULL

11/5/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- make dataframe

ok so the top 50 doesnt work well, what we're gonna do is get the top x tracks across all genres, plays, and get the top artist by plays of each days, top x in most genres, yesterdays, past week, past month (when available). lets also make this just deezer

In [1]:
import argparse
import requests
import pandas as pd
import time
import boto3

In [2]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

test change

In [ ]:
url = 'https://api.deezer.com'

session = requests.Session()
# response = session.get(f'{url}/genre/0/artists', timeout=15)

In [ ]:
# # initial testing
# try:
#     # response = session.get(f'{url}/genre', timeout=15)
#     response.raise_for_status()
#     data_json = response.json()
#     # print('success')
# except requests.RequestException as e: # network, timeout, bad status
#     print("request failed:", e)
# except DeezerError as e:
#     print('failed', e)


# DEFINE FUNCTIONS

gets genre
gets top tracks of each genre
^ adds genre to track listing
gets artist
join in artist fans later in athena

In [ ]:
def get_genre_data() -> pd.DataFrame:
    try:
        response = session.get(f'{url}/genre', timeout=15)
        response.raise_for_status()
        body = response.json() # the whole response
        if "error" in body:
            raise DeezerError('error in genres data body')
        df = pd.DataFrame(body["data"])[['id', 'name']]
        return df[df['id'] != 0] # no 'all' genre
    except requests.RequestException as e:
        print("request failed:", e)
        raise
    except DeezerError as e:
        print("deezer error:", e)
        raise

genres = get_genre_data()

In [40]:
def get_top_tracks() -> pd.DataFrame:
    try:
        result = []
        for genre_id, genre in zip(genres['id'], genres['name']):
            response = session.get(
                f'{url}/chart/{genre_id}/tracks',
                params={'limit': 100},
                timeout=15,)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            df = pd.DataFrame(body['data'])
            df['Genre'] = genre # add in genre data
            result.append(df)
            # print(len(result))
            # time.sleep(.5)
        return pd.concat(result, ignore_index=True)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

top_tracks = get_top_tracks()

In [ ]:
print(top_tracks.shape)
top_tracks.sample(1)

(2700, 17)


,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,preview,md5_image,position,artist,album,type,Genre
1759,3819914,Before I Forget,Before I Forget,,https://www.deezer.com/track/3819914,278,802756,False,0,0,https://cdnt-preview.dzcdn.net/api/1/1/a/d/6/0...,35b093d22fe1539003d5d18dd8f309eb,60,"{'id': 117, 'name': 'Slipknot', 'link': 'https...","{'id': 356130, 'title': 'Vol. 3: The Sublimina...",track,Metal


In [ ]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(top_tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id',
                                                'name':'artist_name',
                                                'link':'artist_link',
                                                'picture':'artist_picture',
                                                'tracklist':'top_tracklist'}, errors='ignore')[['artist_id','artist_name','artist_link','artist_picture','top_tracklist']]
artist_info_df.sample(1)

,artist_id,artist_name,artist_link,artist_picture,top_tracklist
1206,9626504,Luke Combs,https://www.deezer.com/artist/9626504,https://api.deezer.com/artist/9626504/image,https://api.deezer.com/artist/9626504/top?limi...


In [ ]:
album_info_df = pd.json_normalize(top_tracks['album'])
album_info_df = album_info_df.rename(columns={'id':'album_id',
                                              'title':'album_title',
                                              'cover':'album_cover',
                                              'tracklist':'album_tracklist'}, errors='ignore')[['album_id','album_title','album_cover','album_tracklist']]
album_info_df.sample(1)

,album_id,album_title,album_cover,album_tracklist
2158,8088474,King of the Delta Blues SIngers,https://api.deezer.com/album/8088474/image,https://api.deezer.com/album/8088474/tracks


In [49]:
top_tracks = pd.concat([top_tracks, artist_info_df, album_info_df], axis=1).drop(columns=['artist', 'album'])
top_tracks

,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,...,Genre,artist_id,artist_name,artist_link,artist_picture,top_tracklist,album_id,album_title,album_cover,album_tracklist
0,3602074142,Choosin' Texas,Choosin' Texas,,https://www.deezer.com/track/3602074142,232,978085,False,0,0,...,Pop,14867123,Ella Langley,https://www.deezer.com/artist/14867123,https://api.deezer.com/artist/14867123/image,https://api.deezer.com/artist/14867123/top?lim...,837881302,Choosin' Texas,https://api.deezer.com/album/837881302/image,https://api.deezer.com/album/837881302/tracks
1,3895241341,Boston,Boston,,https://www.deezer.com/track/3895241341,170,873262,False,0,0,...,Pop,181157207,STELLA LEFTY,https://www.deezer.com/artist/181157207,https://api.deezer.com/artist/181157207/image,https://api.deezer.com/artist/181157207/top?li...,937796461,Boston,https://api.deezer.com/album/937796461/image,https://api.deezer.com/album/937796461/tracks
2,4250696681,Nicole Kidman,Nicole Kidman,,https://www.deezer.com/track/4250696681,181,990767,True,0,0,...,Pop,306607671,ADÉLA,https://www.deezer.com/artist/306607671,https://api.deezer.com/artist/306607671/image,https://api.deezer.com/artist/306607671/top?li...,1064827231,PRIMA,https://api.deezer.com/album/1064827231/image,https://api.deezer.com/album/1064827231/tracks
3,4078104881,stupid song,stupid song,,https://www.deezer.com/track/4078104881,209,965662,False,0,0,...,Pop,11152580,Olivia Rodrigo,https://www.deezer.com/artist/11152580,https://api.deezer.com/artist/11152580/image,https://api.deezer.com/artist/11152580/top?lim...,1002561451,you seem pretty sad for a girl so in love,https://api.deezer.com/album/1002561451/image,https://api.deezer.com/album/1002561451/tracks
4,4350411742,Housewife,Housewife,,https://www.deezer.com/track/4350411742,208,998094,False,0,0,...,Pop,13817211,Tate McRae,https://www.deezer.com/artist/13817211,https://api.deezer.com/artist/13817211/image,https://api.deezer.com/artist/13817211/top?lim...,1122461572,Housewife,https://api.deezer.com/album/1122461572/image,https://api.deezer.com/album/1122461572/tracks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,38217121,Pa' Que Retozen,Pa' Que Retozen,,https://www.deezer.com/track/38217121,151,629053,True,0,0,...,Latin Music,6729,Tego Calderón,https://www.deezer.com/artist/6729,https://api.deezer.com/artist/6729/image,https://api.deezer.com/artist/6729/top?limit=50,3716991,El Abayarde,https://api.deezer.com/album/3716991/image,https://api.deezer.com/album/3716991/tracks
2696,702421812,QUE PRETENDES,QUE PRETENDES,,https://www.deezer.com/track/702421812,222,676814,False,0,0,...,Latin Music,4860761,J. Balvin,https://www.deezer.com/artist/4860761,https://api.deezer.com/artist/4860761/image,https://api.deezer.com/artist/4860761/top?limi...,101490842,OASIS,https://api.deezer.com/album/101490842/image,https://api.deezer.com/album/101490842/tracks
2697,3721301952,Muévelo,Muévelo,,https://www.deezer.com/track/3721301952,127,685407,False,0,0,...,Latin Music,6043160,Kali Uchis,https://www.deezer.com/artist/6043160,https://api.deezer.com/artist/6043160/image,https://api.deezer.com/artist/6043160/top?limi...,879030832,Muévelo,https://api.deezer.com/album/879030832/image,https://api.deezer.com/album/879030832/tracks
2698,3725958682,Por un Pendejo no se llora,Por un Pendejo no se llora,,https://www.deezer.com/track/3725958682,151,551989,True,0,0,...,Latin Music,11559031,Manuel Turizo,https://www.deezer.com/artist/11559031,https://api.deezer.com/artist/11559031/image,https://api.deezer.com/artist/11559031/top?lim...,880469962,Por un Pendejo no se llora,https://api.deezer.com/album/880469962/image,https://api.deezer.com/album/880469962/tracks


In [57]:
# deduplicated artist list
artist_list = top_tracks['artist_id'].unique().tolist()
len(artist_list)

1716

In [ ]:
# changed, rewrite this to get the artists from top tracks and get their number of fans
def get_artist_data() -> pd.DataFrame:
    try:
        result = []
        for artist_id in top_tracks['artist_id']:
            response = session.get(f'{url}/artist/{artist_id}', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            # display(body)
            # print(type(body))
            result.append(body)
        return result
        # return pd.concat(result, ignore_index=True)[['id','name','Genre']]
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

artists_data = get_artist_data()

In [65]:
artists_df = pd.DataFrame(artists_data)

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

to validate:
- genres
    - not much to do here. check for null data
- top_tracks
    - handle nulls, less than 100 tracks in genre
- artists_df
    - handle null number of fans, missing data in join

# UPLOAD TO S3

In [ ]:
# top_tracks_upload_df = top_tracks
# artists_upload_df = artists_df

# add all this to main so that it's a python script and package